# Reaction Time and Hick's Law
Testing if reaction time increases with log2(number of choices).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

np.random.seed(42)

## Simulating the data
Since I don't have real participant data yet, I simulated reaction times using Hick's Law formula (RT = a + b * log2(N)) plus some random noise, for 5 different choice conditions (1, 2, 4, 8, 16 options), 50 trials each.

In [ ]:
choices_list = [1, 2, 4, 8, 16]
trials = 50

data = []
for n in choices_list:
    log2n = np.log2(n)
    for i in range(trials):
        noise = np.random.normal(0, 40)
        rt = 200 + 150 * log2n + noise
        data.append([n, log2n, rt])

df = pd.DataFrame(data, columns=['choices', 'log2_choices', 'reaction_time'])
df.to_csv('reaction_time.csv', index=False)
df.head()

## Quick look at the data
Average reaction time per choice condition, before any modeling.

In [ ]:
df.groupby('choices')['reaction_time'].mean()

In [ ]:
means = df.groupby('choices')['reaction_time'].mean()
plt.plot(means.index, means.values, marker='o')
plt.xlabel('Number of choices')
plt.ylabel('Average reaction time (ms)')
plt.title('Raw trend before modeling')
plt.show()

## Linear regression
Fitting reaction_time against log2(choices), since that's what Hick's Law actually predicts (not raw choices).

In [ ]:
X = df[['log2_choices']]
y = df['reaction_time']

model = LinearRegression()
model.fit(X, y)

pred = model.predict(X)
r2 = r2_score(y, pred)

print('intercept:', model.intercept_)
print('slope:', model.coef_[0])
print('r2:', r2)

## Plotting the fitted line on the actual data

In [ ]:
x_line = np.linspace(df['log2_choices'].min(), df['log2_choices'].max(), 100).reshape(-1,1)
y_line = model.predict(x_line)

plt.scatter(df['log2_choices'], df['reaction_time'], alpha=0.3, label='trials')
plt.plot(x_line, y_line, color='red', label='regression line')
plt.xlabel('log2(choices)')
plt.ylabel('reaction time (ms)')
plt.title("Hick's Law fit")
plt.legend()
plt.show()

## What this shows
Reaction time goes up as log2(number of choices) goes up, which matches Hick's Law. The slope tells us roughly how many ms are added every time the number of choices doubles. R2 close to 1 here mainly because the data is simulated cleanly - real data will be noisier.